In [ ]:
# CELL 1: Cài dependencies
!pip install -q vllm ragas langchain-openai langchain sentence-transformers qdrant-client

In [ ]:
# CELL 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 3: CONFIG

# Judge LLM — Qwen3.5-27B INT8 via vLLM (local)
JUDGE_MODEL      = "Qwen/Qwen3.5-27B"
JUDGE_PORT       = 8001

# Answer generation model — Qwen3-4B via vLLM (local)
VLLM_MODEL       = "Qwen/Qwen3-4B-Instruct-2507"
VLLM_PORT        = 8000

# Embedding
EMBED_MODEL      = "intfloat/multilingual-e5-large-instruct"

# Qdrant
COLLECTION       = "nutrition_articles"
SNAPSHOT_PATH    = "/content/drive/MyDrive/Nutrition data/nutrition_articles-final.snapshot"
TOP_K            = 10
QUERY_PREFIX     = "Instruct: Tim thong tin dinh duong lien quan\nQuery: "

# File paths
CONTEXTS_FILE    = "/content/drive/MyDrive/Nutrition data/eval_with_contexts_final2.jsonl"
ANSWERS_FILE     = "/content/drive/MyDrive/Nutrition data/eval_answers.jsonl"
RAGAS_SUMMARY    = "/content/drive/MyDrive/Nutrition data/ragas_summary_qwen35.json"
RAGAS_DETAIL     = "/content/drive/MyDrive/Nutrition data/ragas_detail_qwen35.csv"

# Giới hạn số mẫu để test (None = chạy toàn bộ)
RAGAS_SAMPLE_LIMIT = None

In [ ]:
# CELL 4: Khởi động vLLM server cho Qwen3.5-27B (Judge LLM)
# A100 80GB: weights 54GB + KV cache 14GB → max-model-len 32768 hoàn toàn fit
# KV cache Qwen3.5-27B: 144KB/token × 32768 = ~4.7GB << 14GB available

import subprocess, time, requests

LOG_FILE = "/tmp/vllm_judge.log"

with open(LOG_FILE, "w") as log:
    judge_proc = subprocess.Popen(
        [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            "--model",                  JUDGE_MODEL,
            "--port",                   str(JUDGE_PORT),
            "--max-model-len",          "32768",
            "--gpu-memory-utilization", "0.85",
            "--tensor-parallel-size",   "1",
            "--trust-remote-code",
        ],
        stdout=log,
        stderr=log,
    )

print(f"Đang khởi động {JUDGE_MODEL} trên port {JUDGE_PORT}...")
print(f"Log: {LOG_FILE}")
print("Chờ model load — lần đầu download ~54GB có thể mất 15–25 phút\n")

for i in range(240):  # 240 * 10s = 40 phút
    time.sleep(10)

    if judge_proc.poll() is not None:
        print(f"\nProcess đã thoát sớm (exit code {judge_proc.poll()})!")
        print("=== 50 dòng log cuối ===")
        with open(LOG_FILE) as f:
            lines = f.readlines()
        print("".join(lines[-50:]))
        break

    try:
        r = requests.get(f"http://localhost:{JUDGE_PORT}/health", timeout=3)
        if r.status_code == 200:
            print(f"Judge LLM sẵn sàng sau {(i+1)*10}s")
            break
    except:
        pass

    elapsed = (i + 1) * 10
    if elapsed % 60 == 0:
        with open(LOG_FILE) as f:
            lines = f.readlines()
        last = "".join(lines[-3:]).strip() if lines else "(trống)"
        print(f"  {elapsed}s — {last}")
else:
    print("\nTimeout 40 phút — xem log:")
    with open(LOG_FILE) as f:
        lines = f.readlines()
    print("".join(lines[-30:]))

In [ ]:
# CELL 5: Kiểm tra kết nối Judge LLM
from langchain_openai import ChatOpenAI

judge_chat = ChatOpenAI(
    model=JUDGE_MODEL,
    base_url=f"http://localhost:{JUDGE_PORT}/v1",
    api_key="EMPTY",
    temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},  # /no_think
)

try:
    resp = judge_chat.invoke("Xin chào, bạn là mô hình gì?")
    print("Kết nối Judge LLM thành công!")
    print(f"Phản hồi: {resp.content[:200]}")
except Exception as e:
    print(f"Lỗi: {e}")

In [ ]:
# CELL 6: Load embedding model và Qdrant client
import json
from openai import OpenAI
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

embed_model   = SentenceTransformer(EMBED_MODEL)
qdrant_client = QdrantClient(url="http://localhost:6333", timeout=60)
client        = OpenAI(base_url=f"http://localhost:{VLLM_PORT}/v1", api_key="EMPTY")

print(f"Embed model loaded: {EMBED_MODEL}")
print(f"Qdrant client connected")
print(f"vLLM answer client ready at port {VLLM_PORT}")

NUTRITION_SYSTEM_PROMPT = (
    "Ban la chuyen gia tu van dinh duong qua giong noi. Tuan thu:\n"
    "1. Dua vao tai lieu: Tra loi dua tren thong tin dinh duong duoc cung cap.\n"
    "2. Phong cach bac si: Bat dau bang 'Chao ban,', tu van nhu chuyen gia dinh duong.\n"
    "3. Ngan gon, de nghe: Cau tra loi se duoc doc thanh giong noi — dung cau ngan.\n"
    "4. Trung thuc: Neu khong co thong tin → noi ro 'Toi khong co thong tin ve van de nay'.\n"
    "5. Disclaimer: Ket thuc bang 'De duoc tu van chinh xac, ban nen gap bac si dinh duong.'\n"
    "/no_think"
)

def build_prompt(question: str, contexts: list) -> str:
    context_str = "\n\n".join(contexts)
    return (
        f"Tai lieu dinh duong lien quan:\n{context_str}\n"
        f"---\n"
        f"Hay tra loi DUA TREN cac tai lieu tren.\n"
        f"Cau hoi: {question}"
    )

In [ ]:
# CELL 7: RAGAS — chỉ đo Context Recall
# Dùng eval_with_contexts_final2.jsonl (đã có contexts + reference)
# Không cần generate answer — dùng reference làm response

import json, pandas as pd
from ragas import evaluate
from ragas.metrics import ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas import SingleTurnSample, EvaluationDataset
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI

# Judge LLM — Qwen3.5-27B
# max-model-len=32768: input ~5k + output ~4k = ~9k << 32k → không bao giờ bị cắt
_judge_instance = ChatOpenAI(
    model=JUDGE_MODEL,
    base_url=f"http://localhost:{JUDGE_PORT}/v1",
    api_key="EMPTY",
    temperature=0,
    max_tokens=4096,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
_judge_llm = LangchainLLMWrapper(_judge_instance)

metric_recall = ContextRecall()
metric_recall.llm = _judge_llm

# Load data từ eval_with_contexts_final2.jsonl
_results = [json.loads(l) for l in open(CONTEXTS_FILE, encoding="utf-8") if l.strip()]
if RAGAS_SAMPLE_LIMIT:
    _results = _results[:RAGAS_SAMPLE_LIMIT]
    print(f"TEST MODE: chỉ chạy {len(_results)} samples")
else:
    print(f"Chạy toàn bộ {len(_results)} samples")

_samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["reference"],          # dùng reference vì chỉ đo context recall
        retrieved_contexts=r["contexts"],
        reference=r["reference"],
    )
    for r in _results
    if r.get("contexts") and r.get("reference")
]

print(f"Số samples hợp lệ: {len(_samples)}")
print("Bắt đầu chấm Context Recall với Qwen3.5-27B...")

_recall_result = evaluate(
    dataset=EvaluationDataset(samples=_samples),
    metrics=[metric_recall],
    llm=_judge_llm,
    run_config=RunConfig(timeout=180, max_retries=10, max_wait=60, max_workers=2),
)

df_recall = _recall_result.to_pandas()

# Thêm metadata từ input
id_map   = {r["question"]: r.get("id", "")     for r in _results}
src_map  = {r["question"]: r.get("source", "") for r in _results}
df_recall.insert(0, "id",     df_recall["user_input"].map(id_map))
df_recall.insert(1, "source", df_recall["user_input"].map(src_map))

# Lưu toàn bộ ra CSV
out_csv = RAGAS_DETAIL.replace(".csv", "_recall_full.csv")
df_recall.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\nĐã lưu {len(df_recall)} dòng → {out_csv}")

# Thống kê
mean_score = df_recall["context_recall"].mean()
print(f"Context Recall trung bình: {mean_score:.4f}")
print(df_recall["context_recall"].describe().round(4))

bins   = [0, 0.5, 0.7, 0.9, 1.01]
labels = ["<0.5", "0.5–0.7", "0.7–0.9", ">=0.9"]
df_recall["bucket"] = pd.cut(df_recall["context_recall"], bins=bins, labels=labels, right=False)
print("\nSố câu theo bucket:")
print(df_recall["bucket"].value_counts().sort_index())

display(df_recall.drop(columns=["retrieved_contexts", "reference"], errors="ignore"))

In [ ]:
# CELL 8: Lưu kết quả Context Recall
df_recall.to_csv(RAGAS_DETAIL.replace('.csv', '_recall.csv'), index=False)
print(f"Đã lưu → {RAGAS_DETAIL.replace('.csv', '_recall.csv')}")

# Thống kê phân phối
print("\nPhân phối Context Recall:")
print(df_recall['context_recall'].describe())

bins = [0, 0.5, 0.7, 0.9, 1.01]
labels = ['<0.5', '0.5-0.7', '0.7-0.9', '>=0.9']
df_recall['bucket'] = pd.cut(df_recall['context_recall'], bins=bins, labels=labels, right=False)
print("\nSố câu theo bucket:")
print(df_recall['bucket'].value_counts().sort_index())

In [ ]:
# CELL 9: RAGAS đầy đủ (AnswerCorrectness + Faithfulness + ContextPrecision + ContextRecall)
# Cần ANSWERS_FILE (có generated answers từ LLM)

import json
from ragas import evaluate
from ragas.metrics import AnswerCorrectness, Faithfulness, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import SingleTurnSample, EvaluationDataset
from ragas.run_config import RunConfig
from langchain_core.embeddings import Embeddings
from langchain_openai import ChatOpenAI

# Judge LLM — Qwen3.5-27B INT8
llm_instance = ChatOpenAI(
    model=JUDGE_MODEL,
    base_url=f"http://localhost:{JUDGE_PORT}/v1",
    api_key="EMPTY",
    temperature=0,
    max_tokens=2048,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
judge_llm = LangchainLLMWrapper(llm_instance)

class E5Embeddings(Embeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return embed_model.encode(texts, normalize_embeddings=True).tolist()
    def embed_query(self, text: str) -> list[float]:
        return embed_model.encode([text], normalize_embeddings=True)[0].tolist()

judge_embeddings = LangchainEmbeddingsWrapper(E5Embeddings())

metrics = [AnswerCorrectness(), Faithfulness(), ContextRecall(), ContextPrecision()]
for m in metrics:
    m.llm = judge_llm
    if hasattr(m, 'embeddings'):
        m.embeddings = judge_embeddings

results = [json.loads(l) for l in open(ANSWERS_FILE, encoding="utf-8") if l.strip()]
if RAGAS_SAMPLE_LIMIT:
    results = results[:RAGAS_SAMPLE_LIMIT]
    print(f"TEST MODE: chỉ chạy {len(results)} samples")

ragas_samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        retrieved_contexts=r["contexts"],
        reference=r["reference"],
    )
    for r in results
    if r.get("answer") and r.get("contexts") and r.get("reference")
]
dataset = EvaluationDataset(samples=ragas_samples)

print(f"Running full RAGAS ({len(ragas_samples)} samples) với Qwen3.5-27B...")
eval_result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=RunConfig(timeout=180, max_retries=10, max_wait=60, max_workers=2),
)

df_full = eval_result.to_pandas()
df_full.to_csv(RAGAS_DETAIL, index=False)

summary = {
    "judge_model":       JUDGE_MODEL,
    "n_samples":         len(ragas_samples),
    "answer_correctness": round(df_full['answer_correctness'].mean(), 4),
    "faithfulness":       round(df_full['faithfulness'].mean(), 4),
    "context_recall":     round(df_full['context_recall'].mean(), 4),
    "context_precision":  round(df_full['context_precision'].mean(), 4),
}
import json as _json
with open(RAGAS_SUMMARY, 'w') as f:
    _json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n=== RAGAS Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")
display(df_full.head(10))